In [1]:
import math
import os
import pandas as pd
import numpy as np
from classes import GroundSet, MSDAmazonObjective
from greedy_algorithms import greedy, DP_greedy, DP_sample_greedy, random_baseline
from dp_mechanisms import get_best_eps_0


def run_amazon_experiment(objective, ground_set, params, rep):
    results = []
    k, eps, p, lam, g = params['k'], params['eps'], params['private'], params['lambda'], params['gamma']

    # Sensitivity for Amazon Ratings is usually the max rating (e.g., 5.0) / num_users
    delta_target = 1 / (objective.num_users ** 1.5)
    eps_0 = get_best_eps_0(eps_target=eps, delta_target=delta_target, k=k)

    algorithms = [
        ('nonpriv', greedy, [objective, ground_set, k]),
        ('DPGreedy', DP_greedy, [objective, ground_set, k, eps_0, p]),
        ('DPSampleOblGreedy', DP_sample_greedy, [objective, ground_set, k, eps_0, p, True, g]),
        ('DPSampleGreedy', DP_sample_greedy, [objective, ground_set, k, eps_0, p, False, g]),
        ('Random', random_baseline, [objective, ground_set, k])
    ]

    final_selected = {}
    for i in range(rep):
        print(f"\n--- Repetition {i + 1}/{rep} ---")
        for name, func, args in algorithms:
            print(f"Running {name}...")
            res = func(*args)
            selected, value = res[0], res[1]
            queries = res[2] if len(res) > 2 else 0

            results.append({
                'alg': name, 'k': k, 'lambda_param': lam, 'eps': eps,
                'rep': i, 'value': value, 'queries': queries
            })
            final_selected[name] = selected

    df_results = pd.DataFrame(results)
    output_file = "Amazon_Master_Results.csv"
    df_results.to_csv(output_file, mode='a', index=False, header=not os.path.isfile(output_file))

    return final_selected.get('DPGreedy')


In [4]:


# --- Execution ---
# reviews_path = "reviewsHealth_and_Household.csv"
# reviews_df = pd.read_csv(reviews_path, header=None, names=['user_id', 'parent_asin', 'rating', 'timestamp'])
# reviews_df['rating'] = reviews_df['rating']/5.0
# reviews_df = reviews_df.sample(n=100000)
# reviews_df.to_csv('Health_and_Household_example.csv',index=False)
reviews_df = pd.read_csv('Health_and_Household_example.csv')
# meta_path = "Health_and_Household_Final.csv"
# meta_df = pd.read_csv(meta_path, sep='\x1f', low_memory=False)

# meta_df = meta_df[meta_df['categories'].apply(lambda x: 'Health Care' in x)].head(100)
meta_df = pd.read_csv('meta_Health_and_Household_example.csv', sep='\x1f')

# 3. Pre-process Categories into a Dictionary of Sets {asin: {cat1, cat2}}
# This is crucial for the Jaccard distance logic
print("Preparing category lookup...")
product_categories_dict = (
    meta_df.set_index('parent_asin')['categories']
    .astype(str)
    .str.lower()
    .str.split()
    .apply(set)
    .to_dict()
)

# 4. Define the Ground Set (The actual product IDs available to pick from)
all_asins = list(meta_df['parent_asin'].unique())
g_set = GroundSet(elements=all_asins)

param_grid = [
{'k': 4, 'eps': 0.1, 'lambda': 0.015, 'private': False, 'gamma': 0.1},
{'k': 4, 'eps': 0.1, 'lambda': 0, 'private': False, 'gamma': 0.1},
]

for config in param_grid:
    print(
        f"\n================ CONFIG: k={config['k']}, eps={config['eps']}, lam={config['lambda']} ================")
    
    # Initialize Objective with processed data
    obj = MSDAmazonObjective(
        reviews_df=reviews_df,
        product_categories=product_categories_dict,
        lambda_param=config['lambda'],
        k=config['k'],
        distortion=1.0,
    )
    
    run_amazon_experiment(obj, g_set, config, rep=1)  # Reduced reps for speed

Preparing category lookup...

================ CONFIG: k=4, eps=0.1, lam=0.015 ================

--- Repetition 1/1 ---
Running nonpriv...
Iteration 1: Added B0C1G1BJ2B, Total Value: 0.0033
Iteration 2: Added B09JY3BZBS, Total Value: 0.0074
Iteration 3: Added B0C261DBT5, Total Value: 0.0121
Iteration 4: Added B095J4YL2H, Total Value: 0.0183
Running DPGreedy...
Iteration 1: Added B0C1G1BJ2B, Total Value: 0.0033
Iteration 2: Added B09JY3BZBS, Total Value: 0.0074
Iteration 3: Added B0C261DBT5, Total Value: 0.0121
Iteration 4: Added B095J4YL2H, Total Value: 0.0183
Total Value: 0.0254, Coverage: 0.0144, Diversity: 0.7434
Running DPSampleOblGreedy...
Iteration 1: Added B09JY3BZBS, Total Value: 0.0043
Iteration 2: Added B092SW9BX3, Total Value: 0.0072
Iteration 3: Added B0C1G1BJ2B, Total Value: 0.0176
Iteration 4: Added B0C261DBT5, Total Value: 0.0248
Total Value: 0.0248, Coverage: 0.0137, Diversity: 0.7507
Running DPSampleGreedy...
Iteration 1: Added B0C1G1BJ2B, Total Value: 0.0035
Iteration

In [9]:
meta_df.to_csv('meta_Health_and_Household_example.csv', sep='\x1f', index=False)

Another Good Example is simply top k most rated from the "healthcare" category

In [ ]:
df = pd.read_csv("C:\\Users\Ronza\Dev\DP-MSD\\amazon\meta_Health_and_Household.csv", sep='\x1f')
df1 = df[df['categories'].apply(lambda c: 'Health Care' in c)]
df1.sort_values(by='rating_number', ascending=False).head(500)

Or from actual data

In [8]:
df = pd.read_csv("C:\\Users\Ronza\Dev\DP-MSD\\amazon\meta_Health_and_Household.csv", sep='\x1f')
df1 = df[df['categories'].apply(lambda c: 'Health Care' in c)]
df1.sort_values(by='rating_number', ascending=False).head(500)
df['main_category'].value_counts()
df[df['categories'].apply(lambda x: 'Health Care' in x)]
reviews_path = "../../datasets/amazon/FULL_Health_and_Household_Top10k_Dense.csv"
reviews_df = pd.read_csv(reviews_path, header=None, names=['user_id', 'parent_asin', 'rating', 'timestamp'])

# Normalize ratings to [0, 1]
reviews_df['rating'] = reviews_df['rating'] / 5.0

meta_path = "../../datasets/amazon/FULL_meta_Health_and_Household_top10k.csv"
# Using specific separator and limiting to top 200 items by rating count
meta_df = pd.read_csv(meta_path, sep='\x1f', low_memory=False)
meta_df = meta_df.sort_values(by='rating_number', ascending=False)

# --- 2. Constraint Modeling (Partition Matroid) ---
# Constraint: One item per 'store'
partition_map = meta_df.set_index('parent_asin')['store'].fillna('Unknown').to_dict()
unique_stores = meta_df['store'].fillna('Unknown').unique()
partition_limits = {store: 1 for store in unique_stores}

# --- 3. Objective & Ground Set Initialization ---
all_asins = list(meta_df['parent_asin'].unique())

# Log dataset statistics
print(f'Total Reviews: {len(reviews_df)}')
print(f'Unique Users:  {len(reviews_df["user_id"].unique())}')
# --- 1. Find the top 100 most rated items in reviews_df ---
top_100_asins = (
    reviews_df['parent_asin']
    .value_counts()
    .head(100)
    .index.tolist()
)

# --- 2. Filter and Join with meta_df ---
# We use 'inner' join to ensure we only keep items that exist in both 
# the top-rated list AND the metadata file.
meta_df = meta_df[meta_df['parent_asin'].isin(top_100_asins)]

# Log actual count (might be <100 if some ASINs are missing from metadata)
print(f"Items found in metadata: {len(meta_df)}")

# --- 3. Update Constraint Modeling (Partition Matroid) ---
# Re-run these to ensure they match the newly filtered meta_df
partition_map = meta_df.set_index('parent_asin')['store'].fillna('Unknown').to_dict()
unique_stores = meta_df['store'].fillna('Unknown').unique()

# 1. Count exactly how many ratings each ASIN has in reviews_df
count_df = reviews_df['parent_asin'].value_counts().reset_index()
count_df.columns = ['parent_asin', 'ratings_in_review_file']

# 2. Keep the top 100 based on your file's counts
top_100_counts = count_df.head(100)

# 3. Inner join with meta_df to attach store, title, and other details
joined_df = pd.merge(top_100_counts, meta_df, on='parent_asin', how='inner')

# 4. View the result
# Now 'ratings_in_review_file' is the ground truth for your experiment
joined_df[['parent_asin', 'ratings_in_review_file', 'store', 'title', 'categories']].head(100)

Total Reviews: 3208513
Unique Users:  2535881
Items found in metadata: 100


,parent_asin,ratings_in_review_file,store,title,categories
0,B0C1G1BJ2B,59087,Healing Solutions,Healing Solutions 10ml Oils - Cassia Essential...,"['Health & Household', 'Health Care', 'Alterna..."
1,B09JY3BZBS,40895,Penetrex,Penetrex Joint & Muscle Therapy – 2oz Cream – ...,"['Health & Household', 'Health Care', 'Over-th..."
2,B0BCT8SLHR,34466,URPOWER,"URPOWER 2nd Version Essential Oil Diffusers,Ar...","['Health & Household', 'Health Care', 'Alterna..."
3,B0BTXXFVG1,25360,Plant Therapy,Plant Therapy Rosemary Essential Oil 100% Pure...,"['Health & Household', 'Health Care', 'Alterna..."
4,B0813Y7YMV,25343,Artizen,Artizen 4oz Oils - Garlic Essential Oil - 4 Fl...,"['Health & Household', 'Health Care', 'Alterna..."
...,...,...,...,...,...
95,B0C32LR6JR,3878,Sukuos,"Weekly Pill Organizer 7 Day 2 Times a Day, Suk...","['Health & Household', 'Health Care', 'Over-th..."
96,B0BXFP9STJ,3857,UpNature,UpNature Oregano Essential Oil - 100% Natural ...,"['Health & Household', 'Health Care', 'Alterna..."
97,B0B2ZH7YX8,3852,TheraICE Rx,TheraICE Form Fitting Gel Ice Headache Relief ...,"['Health & Household', 'Health Care', 'Over-th..."
98,B006SFUEF2,3813,Munchkin,"Munchkin Nursery Projector and Sound System, W...","['Health & Household', 'Health Care', 'Sleep &..."
